In [1]:
# import required libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# load dataset
df = pd.read_csv("Adm.csv")

# prepare application date
df["Application_DateTime"] = pd.to_datetime(
    df["Application_DateTime"], errors="coerce"
)

# create date features for analysis and prediction
df["Application_Month"] = df["Application_DateTime"].dt.month
df["Application_Year"] = df["Application_DateTime"].dt.year

# remove records without a target value
df = df.dropna(subset=["Chance of Admit"]).copy()

# check dataset
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDataset preview:")
display(df.head())

Dataset shape: (1000, 14)

Missing values:
Serial No.              0
GRE Score               0
TOEFL Score             0
University Rating       0
SOP                     0
LOR                     0
CGPA                    0
Research                0
Chance of Admit         0
Application_DateTime    0
University_Name         0
region                  0
Application_Month       0
Application_Year        0
dtype: int64

Dataset preview:


,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit,Application_DateTime,University_Name,region,Application_Month,Application_Year
0,1,337,118,4,4.5,4.5,9.65,1,0.92,2022-01-01 09:00:00,Indian Institute of Technology Kharagpur,East,1,2022
1,2,324,107,4,4.0,4.5,8.87,1,0.76,2022-01-02 09:00:00,National Institute of Technology Karnataka,South,1,2022
2,3,316,104,3,3.0,3.5,8.00,1,0.72,2022-01-04 09:00:00,Gujarat University,West,1,2022
3,4,322,110,3,3.5,2.5,8.67,1,0.80,2022-01-06 09:00:00,Jamia Millia Islamia,North,1,2022
4,5,314,103,2,2.0,3.0,8.21,0,0.65,2022-01-08 09:00:00,Jamia Millia Islamia,North,1,2022


In [2]:
# select input features and target
features = [
    "GRE Score",
    "TOEFL Score",
    "University Rating",
    "SOP",
    "LOR",
    "CGPA",
    "Research",
    "University_Name",
    "region",
    "Application_Month",
    "Application_Year"
]

X = df[features].copy()
y = df["Chance of Admit"].copy()

# handle missing input values
numeric_features = [
    "GRE Score", "TOEFL Score", "University Rating",
    "SOP", "LOR", "CGPA", "Research",
    "Application_Month", "Application_Year"
]

categorical_features = ["University_Name", "region"]

for col in numeric_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna("Unknown").astype(str)

# encode categorical data
preprocessor = ColumnTransformer([
    ("numeric", "passthrough", numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"),
     categorical_features)
])

# create model pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

# split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# train the model
model.fit(X_train, y_train)

# generate predictions on unseen test data
y_pred = model.predict(X_test)

# calculate evaluation metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RANDOM FOREST REGRESSION PERFORMANCE")
print("-----------------------------------")
print(f"R² Score: {r2:.4f}")
print(f"MAE:      {mae:.4f}")
print(f"RMSE:     {rmse:.4f}")
print(f"Training records: {len(X_train)}")
print(f"Testing records:  {len(X_test)}")

RANDOM FOREST REGRESSION PERFORMANCE
-----------------------------------
R² Score: 0.2306
MAE:      0.0824
RMSE:     0.1149
Training records: 800
Testing records:  200


In [ ]:
# create interactive filters
region_filter = widgets.Dropdown(
    options=["All"] + sorted(df["region"].dropna().unique().tolist()),
    value="All",
    description="Region:"
)

university_filter = widgets.Dropdown(
    options=["All"] + sorted(
        df["University_Name"].dropna().unique().tolist()
    ),
    value="All",
    description="University:"
)

research_filter = widgets.Dropdown(
    options=[
        ("All applicants", "All"),
        ("Research experience", 1),
        ("No research experience", 0)
    ],
    value="All",
    description="Research:"
)

cgpa_filter = widgets.FloatSlider(
    value=float(df["CGPA"].min()),
    min=float(df["CGPA"].min()),
    max=float(df["CGPA"].max()),
    step=0.1,
    description="Minimum CGPA:",
    continuous_update=False
)

chance_filter = widgets.FloatSlider(
    value=0.70,
    min=0.0,
    max=1.0,
    step=0.05,
    description="High chance:",
    continuous_update=False
)

output = widgets.Output()

# dashboard update function
def update_dashboard(region, university, research, min_cgpa, threshold):

    with output:
        clear_output(wait=True)

        # apply filters
        data = df.copy()

        if region != "All":
            data = data[data["region"] == region]

        if university != "All":
            data = data[data["University_Name"] == university]

        if research != "All":
            data = data[data["Research"] == research]

        data = data[data["CGPA"] >= min_cgpa]

        if data.empty:
            display(HTML(
                "<h3>No applicants match the selected filters.</h3>"
            ))
            return

        # calculate KPIs
        total = len(data)
        avg_chance = data["Chance of Admit"].mean() * 100
        high_chance = (data["Chance of Admit"] >= threshold).sum()
        research_pct = data["Research"].mean() * 100

        # dashboard heading
        display(HTML("""
        <h1 style="text-align:center;">
            Admissions Analytics Dashboard
        </h1>
        <p style="text-align:center;">
            Applicant exploration, admissions insights and model evaluation
        </p>
        """))

        # KPI cards
        kpis = [
            ("Total Applicants", f"{total:,}"),
            ("Average Admission Chance", f"{avg_chance:.2f}%"),
            ("High-Chance Applicants", f"{high_chance:,}"),
            ("Applicants with Research", f"{research_pct:.2f}%")
        ]

        cards = "".join([
            f"""
            <div style="
                flex:1;
                min-width:160px;
                padding:18px;
                margin:5px;
                border:1px solid #d6dce5;
                border-radius:10px;
                text-align:center;
                background:#f5f8fc;">
                <div style="font-size:14px;">{title}</div>
                <div style="font-size:25px;font-weight:bold;">
                    {value}
                </div>
            </div>
            """
            for title, value in kpis
        ])

        display(HTML(
            f'<div style="display:flex;flex-wrap:wrap;">{cards}</div>'
        ))

        # chart 1: admission chance distribution
        fig1 = px.histogram(
            data,
            x="Chance of Admit",
            nbins=20,
            color="region",
            title="1. Distribution of Admission Chances",
            hover_data=["CGPA", "GRE Score"]
        )
        fig1.update_layout(template="plotly_white")
        fig1.show()

        # chart 2: applicants by university
        university_data = (
            data.groupby("University_Name")
            .size()
            .reset_index(name="Applicant Count")
            .sort_values("Applicant Count", ascending=False)
            .head(10)
        )

        fig2 = px.bar(
            university_data,
            x="University_Name",
            y="Applicant Count",
            color="Applicant Count",
            title="2. Top 10 Universities by Applicant Count",
            text="Applicant Count"
        )
        fig2.update_layout(
            template="plotly_white",
            xaxis_tickangle=-35
        )
        fig2.show()

        # chart 3: CGPA and admission chance
        fig3 = px.scatter(
            data,
            x="CGPA",
            y="Chance of Admit",
            color="Research",
            size="GRE Score",
            hover_data=[
                "University_Name", "region", "TOEFL Score"
            ],
            title="3. CGPA vs Admission Chance"
        )
        fig3.update_layout(template="plotly_white")
        fig3.show()

        # chart 4: monthly admissions trend
        monthly = (
            data.dropna(subset=["Application_DateTime"])
            .assign(
                Month=lambda d: d["Application_DateTime"]
                .dt.to_period("M").astype(str)
            )
            .groupby("Month", as_index=False)
            .agg(
                Applicants=("Chance of Admit", "size"),
                Average_Chance=("Chance of Admit", "mean")
            )
        )

        if not monthly.empty:
            fig4 = px.line(
                monthly,
                x="Month",
                y="Applicants",
                markers=True,
                title="4. Monthly Applicant Trend"
            )
            fig4.update_layout(template="plotly_white")
            fig4.show()

        # chart 5: research experience comparison
        research_data = data.copy()
        research_data["Research Group"] = research_data["Research"].map({
            0: "No Research",
            1: "Research Experience"
        }).fillna("Other")

        fig5 = px.box(
            research_data,
            x="Research Group",
            y="Chance of Admit",
            color="Research Group",
            title="5. Admission Chance by Research Experience"
        )
        fig5.update_layout(template="plotly_white")
        fig5.show()

        # chart 6: actual versus predicted values
        test_results = pd.DataFrame({
            "Actual Chance": y_test,
            "Predicted Chance": y_pred
        })

        fig6 = px.scatter(
            test_results,
            x="Actual Chance",
            y="Predicted Chance",
            title="6. Model Performance: Actual vs Predicted",
            hover_data=["Actual Chance", "Predicted Chance"]
        )

        fig6.add_trace(go.Scatter(
            x=[0, 1],
            y=[0, 1],
            mode="lines",
            name="Perfect Prediction",
            line=dict(dash="dash")
        ))

        fig6.update_layout(template="plotly_white")
        fig6.show()

        # model performance summary
        metric_df = pd.DataFrame({
            "Metric": ["R² Score", "MAE", "RMSE"],
            "Value": [r2, mae, rmse]
        })

        display(HTML("<h2>Model Performance Summary</h2>"))
        display(metric_df.round(4))

        # feature importance
        encoder = (
            model.named_steps["preprocessor"]
            .named_transformers_["categorical"]
        )

        feature_names = (
            numeric_features
            + list(encoder.get_feature_names_out(categorical_features))
        )

        importances = (
            model.named_steps["regressor"].feature_importances_
        )

        importance_df = pd.DataFrame({
            "Feature": feature_names,
            "Importance": importances
        }).sort_values("Importance", ascending=False).head(12)

        fig7 = px.bar(
            importance_df.sort_values("Importance"),
            x="Importance",
            y="Feature",
            orientation="h",
            title="7. Most Important Model Features"
        )
        fig7.update_layout(template="plotly_white")
        fig7.show()

        # filtered applicant table
        display(HTML("<h2>Applicant Data Exploration</h2>"))

        display(
            data[[
                "Serial No.",
                "GRE Score",
                "TOEFL Score",
                "CGPA",
                "Research",
                "Chance of Admit",
                "University_Name",
                "region"
            ]].sort_values(
                "Chance of Admit", ascending=False
            ).head(20).round(3)
        )

# connect all filters to the dashboard
controls = widgets.interactive(
    update_dashboard,
    region=region_filter,
    university=university_filter,
    research=research_filter,
    min_cgpa=cgpa_filter,
    threshold=chance_filter
)

# display dashboard
display(controls)